# Fault Analysis of Microscaling Formats on a RISC-V SoC

**AIHWS 2026 - artifact reproduction notebook**

Authors: **Dillibabu Shanmugam** and **Patrick Schaumont**, Worcester Polytechnic Institute (WPI), USA.

This notebook reproduces the fault-injection result tables of the paper *"Fault Analysis of
Microscaling Formats on a RISC-V SoC"* **directly from the shipped CSV datasets** in
`../5_datasets`. **No FPGA board, bitstream, or Vivado installation is required** - the raw
voltage-underscaling and clock-glitch campaign logs are consumed offline and the published
numbers are recomputed.

We reuse the verified logic in the companion script `reproduce.py` (same directory), which is
the reference implementation used to generate the paper's tables:

- **Table 7** - QNN voltage underscaling: per-format Normal / Faults / Crashes / Dead counts,
  plus fault onset, crash onset, and shared-exponent (SE) corruption onset voltages.
- **Table 8** - BitNet FC1 voltage underscaling: per-format outcome counts and SE onset.
- **Table 10** - Fault-injection method comparison: voltage underscaling vs clock glitching.
- **Figure 5** - design-space fault-resilience ordering (silent-fault count per format).

Outcome taxonomy (paper Sec. 3.2), read from the `class` column of each CSV:
`normal | fault` (silent data corruption, SDC) `| crash | dead`. SE corruption is flagged when
`se_a != 127` on a faulted row (127 is the neutral shared-exponent sentinel).


In [1]:
# Reproduce Tables 7, 8, and 10 by importing the verified logic from reproduce.py.
# reproduce.py reads the CSVs in ../5_datasets (columns: format, vccint, class, se_a,
# ret, golden_ret) and recomputes every published value; here we call its functions
# directly and render the same Markdown tables it emits on the command line.
import os, sys

sys.path.insert(0, os.path.abspath("."))
from reproduce import qnn_table, bitnet_table, method_comparison, render, fmt_onset

# render() prints Table 7, Table 8, Table 10, and the Fig. 5 fault-resilience ordering.
render()


## Table 7 - QNN voltage underscaling (570 experiments, 5 formats)

| Format | Normal | Faults | Crashes | Dead | Fault onset | Crash | SE onset |
|--------|-------:|-------:|--------:|-----:|:-----------:|:-----:|:--------:|
| MXFP8-E5M2 | 57 | 27 | 27 | 3 | 0.81 V | 0.77 V | 0.68 V |
| MXFP8-E4M3 | 60 | 24 | 27 | 3 | 0.80 V | 0.77 V | 0.68 V |
| MXINT8 | 66 | 18 | 27 | 3 | 0.78 V | 0.77 V | 0.68 V |
| LOG8-SUM | 69 | 15 | 27 | 3 | 0.68 V | 0.77 V | 0.68 V |
| LOG8-MAX | 69 | 15 | 27 | 3 | 0.68 V | 0.77 V | 0.68 V |

## Table 8 - BitNet FC1 voltage underscaling (570 experiments, 5 formats)

| Format | Normal | Faults | Crashes | Dead | SE onset | Golden ret |
|--------|-------:|-------:|--------:|-----:|:--------:|-----------:|
| MXFP8-E5M2 | 71 | 15 | 25 | 3 | 0.68 V | 0  (golden=0 masks CPU SDC) |
| MXFP8-E4M3 | 71 | 14 | 26 | 3 | 0.68 V | 0  (golden=0 masks CPU SDC) |
| MXINT8 | 63 | 22 | 26 | 3 | 0.68 V | 23 |
| LOG8-SUM | 66 | 19 | 26 | 3 | 0.68 V | 10 |
| LOG8-MAX | 66 | 21 | 24

In [2]:
# Figure 5 - QNN fault-resilience ordering: horizontal bar chart of the silent-fault
# (SDC) count per MX format, fewest = most resilient. Values are pulled live from the
# reproduced Table 7 (qnn_table) so the chart cannot drift from the data.
import matplotlib
matplotlib.use("Agg")  # non-interactive backend so this runs headless
import matplotlib.pyplot as plt

# (format, silent-fault count) sorted most-resilient (fewest faults) at the top.
ranked = sorted(qnn_table(), key=lambda r: r[1][1])
names = [name for name, cnts, *_ in ranked]
silent = [cnts[1] for name, cnts, *_ in ranked]

# Sanity check against the paper's Fig. 5 silent-fault counts.
expected = {
    "LOG8-SUM": 15, "LOG8-MAX": 15, "MXINT8": 18,
    "MXFP8-E4M3": 24, "MXFP8-E5M2": 27,
}
for name, cnts, *_ in ranked:
    assert cnts[1] == expected[name], (name, cnts[1], expected[name])
print("Silent-fault counts match paper Fig. 5:", dict(zip(names, silent)))

fig, ax = plt.subplots(figsize=(8, 4.2))
ypos = range(len(names))
# Draw most-resilient at the top of the axis.
bars = ax.barh(list(ypos), silent, color="#4C72B0", edgecolor="black", height=0.6)
ax.set_yticks(list(ypos))
ax.set_yticklabels(names)
ax.invert_yaxis()  # first entry (LOG8-SUM, fewest faults) on top
ax.set_xlabel("Silent faults (SDC count) - lower is more resilient")
ax.set_title("Figure 5: QNN MX-format fault resilience ordering")
for b, v in zip(bars, silent):
    ax.text(b.get_width() + 0.3, b.get_y() + b.get_height() / 2,
            str(v), va="center", ha="left", fontweight="bold")
ax.margins(x=0.12)
fig.tight_layout()
fig.savefig("figure5_fault_resilience.png", dpi=120)
plt.show()
print("Saved figure5_fault_resilience.png")


Silent-fault counts match paper Fig. 5: {'LOG8-SUM': 15, 'LOG8-MAX': 15, 'MXINT8': 18, 'MXFP8-E4M3': 24, 'MXFP8-E5M2': 27}
Saved figure5_fault_resilience.png


<ipython-input-2-0fd1136cf783>:37: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Expected published values (for confirmation)

The reproduced tables above should match the paper exactly. Reference values:

**Table 7 - QNN voltage underscaling** (Normal / Faults / Crashes / Dead, fault onset):

| Format | Normal | Faults | Crashes | Dead | Fault onset |
|--------|:------:|:------:|:-------:|:----:|:-----------:|
| MXFP8-E5M2 | 57 | 27 | 27 | 3 | 0.81 V |
| MXFP8-E4M3 | 60 | 24 | 27 | 3 | 0.80 V |
| MXINT8     | 66 | 18 | 27 | 3 | 0.78 V |
| LOG8-SUM   | 69 | 15 | 27 | 3 | 0.68 V |
| LOG8-MAX   | 69 | 15 | 27 | 3 | 0.68 V |

**Figure 5 - silent-fault ordering** (fewest = most resilient):
LOG8-SUM = 15, LOG8-MAX = 15, MXINT8 = 18, MXFP8-E4M3 = 24, MXFP8-E5M2 = 27.

**Table 10 - method comparison:** voltage underscaling total 1140 -> 262 crashes / 190 silent
faults (SDC); clock glitching total 900 -> 0 crashes / 0 silent faults.

If the cells above print these numbers, the offline reproduction from the CSV datasets is
confirmed - no FPGA hardware required.
